# 3.1 — is dilation the reason?

`dilated-style-64-dihedral8` scored **0.8995** [0.8856, 0.9104] with `Scratch` at **0.793** —
the best numbers in the project, above `v27-resnet_style`'s 0.8900. Two things make it
provisional:

1. It ran with **`dihedral8`**, not the settled `rotation`. Nothing else in v27–v30 uses
   dihedral8, and rotation beat it on `baseline_cnn` (0.8800 vs 0.8587).
2. Its best epoch was **45 of 50** — still improving when the cap stopped it. 0.8995 is a floor.

Four arms. **Dilation costs no parameters** — `padding = dilation * (kernel // 2)` holds the
output shape, so every arm below has exactly the same parameter count as its control and
differs only in receptive field:

| arm | model | params | dilation | control |
|---|---|---|---|---|
| `11_dilated_rotation_64` | `dilated_style` | 298,377 | (1,2,4,8)x2, field 63x63 | `12` |
| `12_dilated_control_64` | `dilated_style` | 298,377 | all 1, field 19x19 | — |
| `13_convnext_dilated_128` | `convnext_style` | 2,675,849 | (1,2,4) | `v28-convnext_big_128` |
| `14_resnet_dilated_64` | `resnet_style` | 2,829,097 | (1,1,2,4) | `v27-resnet_style` (0.8900) |

## Why 12 is the arm that matters

Same architecture, same depth, same 298,377 parameters, same stack that never downsamples —
the *only* difference is that the kernels are spaced out. If 11 beats 12, dilation is the
cause and nothing else can be. If they tie, the dilated model's advantage is its
full-resolution stack rather than the spacing, which is still a real finding and a different
sentence in the presentation.

## Why the rates are not uniform

Not every layer needs dilation. In `13` and `14` the early stages stay undilated, where the
map still holds the resolution a one-die-wide `Scratch` needs, and only the late stages
widen their context once the map is 32x32 or smaller and context is what is missing.

## Why these architectures

`v28-convnext_big_128` reached `Scratch` **0.804** by epoch 28, the best seen, and
`v27-resnet_style` tops the v27 table at 0.8900. Adding dilation to models that are already
winning tests whether it adds something, rather than whether it rescues something weak.

`max_epochs` is 100 on every arm — the previous dilated run truncated, and these are being
given room. `BUDGET_HOURS` is off.

## 0. Colab web UI only — clone and authenticate

Skip if the repo is already at `/content/fdl-project`.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Run

Cheapest-first. Each arm prints its dilation rates and receptive field before training, so a
config that lost them stops being a mystery an hour later.

`13` and `14` need the dilation support added to `convnext_style` and `resnet_style` in this
same commit (`src/fdl_project/models/`, tests in `tests/test_dilation.py`) — so **pull before
running**, or those two arms will fail on an unknown `dilation` keyword.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

SERIES = "v31_dilated"
RERUN = False
BUDGET_HOURS = None          # no cap: every arm here is worth its two hours

CONFIG_DIRECTORY = REPO / "configs/train" / SERIES
CONFIGS = sorted(CONFIG_DIRECTORY.glob("*.yaml"))
assert CONFIGS, f"no configs in {CONFIG_DIRECTORY} -- pull the branch"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},dilation]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) done")

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)
session_started = time.monotonic()

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"
    assert config.trainer.max_epochs == 100, "these arms need room; the last one truncated"

    if config.name in done:
        print(f"skip  {config.name}")
        continue
    if BUDGET_HOURS is not None and (time.monotonic() - session_started) / 3600 > BUDGET_HOURS:
        print(f"\nbudget reached -- stopping before {config.name}")
        break

    model = build_model(config.model.name, **config.model.kwargs)
    parameters = count_trainable_parameters(model)
    rates = getattr(model, "dilation_rates", None)
    field = getattr(model, "receptive_field", None)
    del model
    print(f"\n=== {config.name}  ({config.model.name}, {parameters:,} parameters)")
    print(f"    dilation {rates}"
          + (f", receptive field {field}x{field}" if field else "")
          + f", {config.data.preprocessing.target_size[0]}px")

    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
    results.append({
        "run": config.name,
        "model": config.model.name,
        "dilation": str(rates),
        "px": config.data.preprocessing.target_size[0],
        "params": parameters,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "scratch_f1": round(float(per_class["Scratch"]), 3),
        "loc_f1": round(float(per_class["Loc"]), 3),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    row = results[-1]
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"Scratch {row['scratch_f1']:.3f}  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete -> {RESULTS_CSV}")

## 4. Read it as matched pairs

Every comparison here holds the parameter count fixed, so a difference is attributable to
dilation and nothing else. Noise floor **0.02**.

In [ ]:
# Every one of these is a matched control: dilation costs no parameters, so each
# pair below differs in the dilation rates and nothing else.
CONTROLS = {
    "v31-dilated_rotation_64":   ("v31-dilated_control_64", None),
    "v31-convnext_dilated_128":  ("v28-convnext_big_128",   None),   # filled below if measured
    "v31-resnet_dilated_64":     ("v27-resnet_style",       0.8900),
}
REFERENCE = {                    # measured on the same splits and pipeline
    "dilated-style-64-dihedral8 (old aug)": 0.8995,
    "v27-resnet_style":                     0.8900,
    "v27-convnext_style":                   0.8883,
    "v28-convnext_big_64":                  0.8862,
    "v27-baseline_cnn":                     0.8646,
}
NOISE_FLOOR = 0.02

frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2
pd.set_option("display.width", 250)
display(frame)

scored = frame.set_index("run")["macro_f1"]
print(f"Dilation, matched controls (noise floor {NOISE_FLOOR:.2f}):")
for arm, (control, known) in CONTROLS.items():
    if arm not in scored.index:
        print(f"  {arm:28} not run yet")
        continue
    reference = scored.get(control, known)
    if reference is None:
        print(f"  {arm:28} {scored[arm]:.4f}  (control {control} not measured yet)")
        continue
    delta = scored[arm] - reference
    verdict = "REAL" if abs(delta) > NOISE_FLOOR else "noise"
    print(f"  {arm:28} {delta:+.4f} vs {control} ({reference:.4f})  [{verdict}]")

print("\nFor context, on the same splits:")
for label, score in REFERENCE.items():
    print(f"  {label:38} {score:.4f}")

if frame["truncated"].any():
    print("\nStill improving at the cap:", ", ".join(frame.loc[frame["truncated"], "run"]))

## 5. What it settles

* **`11` > `12`** — dilation earns its place, and the project's best model has a mechanism
  behind it rather than a lucky draw.
* **`11` == `12`** — the win belongs to never downsampling, not to the spacing. Then the
  presentation's story is about **resolution**, which also explains `v28-convnext_big_128`'s
  `Scratch` 0.804 and phase 2's 224px result pointing the same way.
* **`13`/`14` > their controls** — dilation transfers to architectures that were already
  winning, which is the stronger claim of the two.

Whichever holds, `Scratch` is the column to watch: it is the thinnest pattern, the weakest
class, and the one the whole full-resolution argument is about.